---
title: "Internes Szenario- und Kalibrierungsdashboard"
execute:
  echo: true
format:
  html:
    code-fold: true
---

# Interner Arbeitsstand {#sec-internes-dashboard}

::: {.callout-warning title="Dashboard nach Bestandsanalyse erneuern"}
**Aktualisierungsbedarf · 09.11.2026**

Die vollständige Flächenlieferung zunächst analysieren und bestand.parquet erzeugen. Danach Bestand heute oder 2040 wählen, Dashboard ausführen und PEQ-Bewertungsflächen auswählen. Die bisherige feste 20-%-Startauswahl entfällt; die nationale Referenz 2040 ist beim heutigen Bestand kein zeitgleicher Vergleich.
:::

**Nur für die interne Redaktion. Kein Bestandteil der finalen Publikation.**
Das Profil `publikation` schließt diesen Anhang aus. Das Dashboard verändert weder
die wissenschaftlichen Referenznotebooks noch deren Ergebnisdateien oder QMD-Texte.

**Bedienung:** Zuerst das Gebäudesektormodell mit den gewünschten Flächenregeln ausführen. Danach hier nationales Szenario wählen,
Kernel neu starten und alle Zellen ausführen. Danach HTML neu rendern. Die HTML-Regler
rechnen unmittelbar mit diesem eingebetteten Datenstand; sie können keine lokale Excel
neu einlesen. Der JSON-Download sichert Parameter, Ergebnisse und Eingabe-Hashes.
Für eine Übernahme in den Bericht müssen die Referenznotebooks bewusst aktualisiert werden.

Quellen und Bilanzgrenzen: @sec-gebaeudebestand und @sec-allokation sowie die
wissenschaftlichen Anhänge. Verkehrsanteile beziehen sich auf **elektrische Energie**,
nicht auf Fahrten oder Personenkilometer. Alle Salden: Erzeugung minus Bedarf.

**Bestandswahl:** BESTAND wählt heute oder 2040 aus der Analyse. Die nationale Referenz ist ausschließlich 2040; beim heutigen Bestand ist sie nur eine Zukunftsreferenz, kein zeitgleicher Kalibrierungsvergleich. Neue Analysedaten stehen erst nach Lieferung der vollständigen Flächentabelle zur Verfügung.


In [ ]:
from pathlib import Path
import json, hashlib, sys
import numpy as np
import pandas as pd
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'_quarto.yml').exists())
BASE = ROOT/'Berechnungen/gebaeudesektormodell'
sys.path.insert(0, str(BASE))
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

# Ausgangsszenario: Original-Pool (Strombed_estim + zuordnung_gfz)
# oder editierbare Reinschrift (Typen + GFZ_Zuordnung).
XLSX = BASE/'data/Building-Sector_Pool_1.xlsx'
BESTAND = '2040'  # alternativ: 'heute'
RESULTS = BASE/'outputs/analyse/bestand.parquet'
SCENARIO = 'transition_2040'
AREA_COLUMN = 'Nationale BGF 2040'
# Andere Flächenjahre sind kein zeitlich konsistenter Vergleich mit UBA 2040!
DEFAULT = dict(miv=.8, other_transport=0., free_pv=0.,
               saving=0., pv_scale=1., ngf_bgf=.8,
               density_f_res=30.4, density_f_ed=26.4, density_dx=.15, density_cutoff=62.)
# saving: proportionale Strombedarfssenkung des gewählten Gebäudepools.
# pv_scale: unabhängige Skalierung seiner gesamten PV-Erzeugung, keine PEExcel-Neusimulation.
# Freiflächen-PV gehört vollständig zu den zentralen Großkraftwerken.


## Eingaben, Abdeckung und Herkunft

Das Dashboard liest die gemeinsame Ergebnisdatei des Gebäudesektormodells; nur die
GFZ-Zuordnung wird ergänzend aus der Basisflächen-Excel übernommen. Variantenflächen
werden hier nicht nochmals berechnet. BGF in m², Energiekennwerte in kWh/(m² NGF·a).
Teilweise als PEQ zugeordnete Varianten werden in PEQ- und Restflächen aufgeteilt,
damit die interaktive Auswahl nicht unbeabsichtigt die gesamte Variantenfläche umfasst.


In [ ]:
data=pd.read_parquet(RESULTS)
data=data.loc[data.Szenario.eq(BESTAND)].copy()
if data.empty: raise ValueError('Der gewählte Bestand fehlt; Analyse mit vollständiger Flächentabelle ausführen.')
mapping=pd.read_excel(XLSX,sheet_name='zuordnung_gfz',usecols='A:F')
assert data.Archetype.is_unique
area=data.BGF_m2
assert area.notna().all() and area.ge(0).all()
data['BGF']=area
data['demand']=data.Strombedarf
data['pv']=data.PV_Erzeugung
national = {}
source_files = [XLSX, RESULTS, RESULTS.parent/'validation.json']
for kind,filename in [('g','stromerzeugung_2040.csv'),('d','strombedarf_2040.csv')]:
    path=ROOT/'Daten/sektorenmodell'/filename
    table=pd.read_csv(path)
    selected=table.loc[table.scenario.eq(SCENARIO)]
    assert len(selected) and selected.metric.is_unique and selected.value_pj.notna().all()
    assert selected.year.eq(2040).all()
    national[kind]=selected.set_index('metric').value_pj.to_dict()
    source_files.append(path)
    display(selected[['metric','value_pj','source_id','source_location']])
g,d=national['g'],national['d']
display(pd.DataFrame([{'Datei':str(XLSX),'Flächenspalte':AREA_COLUMN,'Typen':len(data),
 'positive Typen':int(area.gt(0).sum()),'BGF [Mio. m²]':area.sum()/1e6,'SHA256':sha256(XLSX)}]))


## Dichtezuordnung und Erwartungswert

Ein einziges Ziel erhält 100 %. Bei zwei Zielen müssen beide Anteile vorhanden sein
und zusammen 100 % ergeben. Identische zusammengefasste Statistiknutzungen werden
nicht doppelt gezählt. Die GFZ-Verteilung 2021 bleibt ein vorläufiges räumliches Muster.

$$E[c_D\mid k]=\sum_j s_{kj}\,\left[-\min\left(\frac{f_{RES}}{GFZ_j+dx}-f_{ED},c_{max}\right)\right].$$

$s_{kj}$ ist der bedingte BGF-Anteil im GFZ-Bin, $k$ die Geometrieklasse.
Die historische Kurve (Schneider et al. 2023, Tabelle 5/Gleichung 7) wird wie im
wissenschaftlichen Anhang vorläufig als EE-Strom interpretiert. Sie ist ein
**virtueller Bewertungswert**, keine Stromerzeugung und keine bestätigte neue Kalibrierung.


In [ ]:
use_map={'Gebäude für Gemeinschaften':'Gemeinschaftsgebäude','Hotels und ähnliche Gebäude':'Hotels o.Ä.',
 'Bürogebäude':'Büronutzung','Groß- und Einzelhandelsgebäude':'Groß- und Einzelhandelsgebäude',
 'Gebäude des Verkehrs- und Nachrichtenwesens':'Verkehrs- und Nachrichtenwesen','Industrie- und Lagergebäude':'Industrie und Lagerung',
 'Gebäude für Kultur- und Freizeitzwecke sowie des Bildungs- und Gesundheitswesens':'Kultur, Freizeit, Bildung und Gesundheit',
 'Wohngebäude mit einer Wohnung':'Ein- und Zweifamilienhäuser','Wohngebäude mit zwei Wohnungen':'Ein- und Zweifamilienhäuser',
 'Wohngebäude mit drei und mehr Wohnungen':'Mehrfamilienhaus'}
mapping['GWR_Nutzung']=mapping.Nutzung.map(use_map); assert mapping.GWR_Nutzung.notna().all()
mapping['GK']=mapping.Geschoss.str.replace(' Geschoße','',regex=False).str.replace(' Geschoß','',regex=False).replace({'6 bis 10':'6-10','11 und mehr':'11+'})
mapping['Einzelziel_ergänzt']=mapping.Anteil1.isna() & mapping.Archetype1.notna() & mapping.Archetype2.isna()
if True:
    mapping.loc[mapping['Einzelziel_ergänzt'],'Anteil1']=1.0
# Doppelte Statistiknutzungen nur zusammenführen, wenn ihre Regeln identisch sind.
for key,group in mapping.groupby(['GWR_Nutzung','GK']):
    assert len(group[['Anteil1','Archetype1','Anteil2','Archetype2']].drop_duplicates())==1,key
unique=mapping.drop_duplicates(['GWR_Nutzung','GK'])
long=[]
for _,row in unique.iterrows():
    for n in [1,2]:
        if pd.notna(row[f'Archetype{n}']):
            long.append({'GWR_Nutzung':row.GWR_Nutzung,'GK':row.GK,'Geometrie':row[f'Archetype{n}'],'Anteil':row[f'Anteil{n}']})
links=pd.DataFrame(long)
assert links.Anteil.notna().all(), 'GFZ-Anteile offen; Einzelzielregel oder Eingaben prüfen'
assert links.Anteil.between(0,1).all()
np.testing.assert_allclose(links.groupby(['GWR_Nutzung','GK']).Anteil.sum(),1)
gfz=pd.read_csv(ROOT/'Berechnungen/gfz_at/outputs/recalculated/gfz_distribution_by_province_use_storeys.csv',dtype={'GK':str})
joined=gfz.merge(links,on=['GWR_Nutzung','GK'],how='left',indicator=True)
assert joined['_merge'].eq('both').all()
joined['zugeordnete_BGF']=joined.BGF_m2*joined.Anteil
np.testing.assert_allclose(joined.zugeordnete_BGF.sum(),gfz.BGF_m2.sum(),rtol=1e-12)
bins=joined.groupby(['Geometrie','GFZ'],as_index=False).zugeordnete_BGF.sum()
bins['s']=bins.zugeordnete_BGF/bins.groupby('Geometrie').zugeordnete_BGF.transform('sum')
assert set(data.loc[data.BGF.gt(0),'Geometrie']).issubset(set(bins.Geometrie)), 'GFZ-Zuordnung fehlt'
source_files.append(ROOT/'Berechnungen/gfz_at/outputs/recalculated/gfz_distribution_by_province_use_storeys.csv')
display(links)


## Rechenmodell und Bilanzkontrollen

$$D_B=\sum_k A_k r e_{el,k}(1-u),\quad P_B=\sum_k A_k r e_{PV,k}v.$$

Gebäude-PV enthält Eigenverbrauch und Einspeisung. Freiflächen-PV gehört zu den zentralen Großkraftwerken. H₂ (4 PJ/a) und fossiler Strom (1 PJ/a) sind vollständig enthalten. Als dokumentierte Modellannahme vom 29.10.2026 gelten 5 statt 7 PJ/a Nettoexport; Quelldaten bleiben unverändert.

Die zentrale Restdeckung $R$ nach allen Bedarfen außerhalb Gebäude + MIV wird ohne MIV-Obergrenze zugeordnet. Negative Reste werden als zentraler Fehlbetrag sichtbar und durch Gebäude-PV ausgeglichen.

$$T_B=P_{PV,TD}(1-f)-D_{B,TD},\quad T_{B+MIV}=T_B+R-D_{MIV}=0.$$

Bottom-Up und Top-Down verwenden dieselbe Restdeckung. Ihre Differenz ist $\Delta=(P_B-D_B)-T_B$. Dichtewerte bleiben separat; sie schließen keine physische Energielücke.


In [ ]:
generation_keys=['hydro','wind','biomass','geothermal','hydrogen_electricity','pv','fossil']
demand_keys=['verkehr','gebaeude','industrie','landwirtschaft','verbrauch_sektor_energie','transportverluste','umwandlungseinsatz']
residual=sum(g[k] for k in generation_keys)+g['net_imports']-sum(d[k] for k in demand_keys)
assert SCENARIO == 'transition_2040', 'Exportannahme nur für Transition 2040'
assumptions=dict(date='2026-10-29',source_net_exports_pj=7,model_net_exports_pj=5,export_adjustment_pj=-2,
    reason='Modellannahme zur Bilanzschließung',hydrogen_electricity_included_pj=4,fossil_included_pj=1)
model_net_imports=-assumptions['model_net_exports_pj']
model_residual=residual+model_net_imports-g['net_imports']
# Aggregation in PJ bei r=1; der editierbare Flächenfaktor wird erst im Dashboard angewandt.
# Teilflächen getrennt halten: PEQ-Auswahl darf nicht die ganze Variante markieren.
pieces=[]
for selected in [True,False]:
 part=data.copy()
 fraction=part.PEQ_Anteil_2040 if selected else 1-part.PEQ_Anteil_2040
 part['BGF']=part.BGF*fraction
 part['default_selected']=selected
 part['Archetype']=part.Archetype+('::PEQ' if selected else '::Rest')
 pieces.append(part.loc[part.BGF.gt(0)])
active=pd.concat(pieces,ignore_index=True)
active['D']=active.BGF*active.demand*3.6e-9
active['P']=active.BGF*active.pv*3.6e-9
geometry=active.groupby('Geometrie').agg(BGF=('BGF','sum'),D=('D','sum'),P=('P','sum'))
def calculate(p):
    assert all(np.isfinite(v) for v in p.values())
    assert all(0<=p[k]<=1 for k in ['miv','other_transport','free_pv','saving'])
    assert p['miv']+p['other_transport']<=1
    assert p['ngf_bgf']>0 and p['pv_scale']>=0 and p['density_dx']>0
    assert all(p[k]>=0 for k in ['density_f_res','density_f_ed','density_cutoff'])
    central=sum(g[k] for k in generation_keys if k!='pv')+g['pv']*p['free_pv']+max(model_net_imports,0)
    fixed=sum(d[k] for k in ['industrie','landwirtschaft','verbrauch_sektor_energie','umwandlungseinsatz','transportverluste'])+max(-model_net_imports,0)
    raw=central-fixed-d['verkehr']*(1-p['miv'])
    miv=d['verkehr']*p['miv']; credit=raw
    target=g['pv']*(1-p['free_pv'])-d['gebaeude']
    demand=geometry.D.sum()*p['ngf_bgf']*(1-p['saving'])
    pv=geometry.P.sum()*p['ngf_bgf']*p['pv_scale']
    closure=target+credit-miv
    assert np.isclose(closure,model_residual)
    return dict(demand=demand,pv=pv,balance=pv-demand,target=target,credit=credit,
                extended=pv-demand+credit-miv,target_extended=target+credit-miv,
                gap=pv-demand-target,raw=raw,closure=closure)
reference=calculate(DEFAULT)
for m,f in [(0,0),(1,1),(.6,.2)]:
    result=calculate(dict(DEFAULT,miv=m,free_pv=f))
    assert np.isclose(result['extended']-result['target_extended'],result['gap'])
display(pd.DataFrame([reference],index=['Referenz; PJ_el/a']).T)
payload=dict(defaults=DEFAULT,reference=reference,g=g,d=d,residual=residual,model_residual=model_residual,model_net_imports=model_net_imports,assumptions=assumptions,schema_version=2,
             geometry=geometry.reset_index().to_dict('records'),
             bins=bins[['Geometrie','GFZ','s']].to_dict('records'),
             provenance={str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else str(p):sha256(p) for p in source_files},
             scenario=SCENARIO,bestand=BESTAND,area_column='BGF_m2',xlsx=str(XLSX))


## Selektive Dichtebewertung über die GFZ-Verteilung

Für Typ $k$, Nutzung $u$ und GFZ-Klasse $j$ ist $s_{kuj}$ der zugeordnete BGF-Anteil, mit $\sum_{u,j}s_{kuj}=1$. Die Verteilung stammt aus dem GFZ-Modell 2021 und wird innerhalb der zugeordneten Geometrieklasse auf die Szenarioflächen übertragen. Energiekennwerte eines Typs sind innerhalb dieser Verteilung konstant; eine GFZ-abhängige PV-Neusimulation findet nicht statt.

$$P_{uj}=\sum_k s_{kuj} A_k r e_{PV,k}v,\quad D_{uj}=\sum_k s_{kuj} A_k r e_{el,k}(1-u_s).$$

$A_k$ ist die BGF in m², $r$ das NGF/BGF-Verhältnis, $v$ die PV-Skalierung und $u_s$ die zusätzliche Bedarfssenkung. Die Kennwerte beziehen sich auf m² NGF. Für die ausgewählte Typenmenge $K_{PEQ}$ wird die virtuelle Korrektur ausschließlich auf deren BGF angewendet:

$$T_{uj}=\sum_{k\in K_{PEQ}} s_{kuj} A_k t(GFZ_j),\quad C_{uj}=T_{uj}-(P_{uj,PEQ}-D_{uj,PEQ}),\qquad B_{\leq j}=\sum_{i\leq j,u}(P_{ui}-D_{ui}),\quad B^*_{\leq j}=B_{\leq j}+\sum_{i\leq j,u}C_{ui}.$$

Energie in kWh wird mit $3{,}6\cdot10^{-9}$ in PJ umgerechnet. Die kumulierte Linie summiert Klassenbeiträge von niedriger zu hoher GFZ; sie ist kein spezifischer Kennwert. Ihr Endpunkt entspricht der Sektorsumme ohne Mobilität. Die unbereinigte GFZ-Aggregation muss die direkte BGF-Bilanz reproduzieren. Die Korrektur bleibt separat schraffiert und verändert keine physische Stromerzeugung. Startauswahl und einzelne Typen sind im nachgelagerten Dashboard editierbar; die vollständige GFZ-Spanne bleibt erhalten.


Der Zielsaldo ist $t(GFZ)=\min(f_{RES}/(GFZ+dx)-f_{ED},c_{max})$. Die negative historische Bewertungskorrektur prüft die Zielerfüllung; sie wird hier nicht zusätzlich auf den Gebäudesaldo addiert. Stattdessen ersetzt der Zielsaldo den typologischen Saldo ausgewählter Typen. Die schraffierte Anpassung $C$ kann durch mehr Erzeugung oder weniger Bedarf erreicht werden. Nicht ausgewählte Typen bleiben unverändert. PEQ-Typen sind in beiden Grafiken grün; sonstige Typen behalten die Nutzungsfarben. Die Ansicht GFZ 0–5 begrenzt nur die Darstellung, nicht die Berechnung.

Die spezifische Klassenbilanz trennt Nicht-PEQ, ausgewählte PEQ mit typologischen Kennwerten und dieselben PEQ bei Erreichen des Dichteziels. Jede Gruppe wird durch ihre eigene BGF innerhalb der Geometrieklasse geteilt. Die drei spezifischen Salden sind nicht addierbar; leere Gruppen bleiben ohne Wert. Die Verteilungsgrafik verwendet BGF-gewichtete Quantile der Zielwerte: Das q-Quantil ist der kleinste Zielwert, dessen kumuliertes Flächengewicht mindestens q erreicht. Der gewichtete Mittelwert ergänzt Median und Perzentilband. Top-Down, Typologie und Dichteausgleich werden ohne Mobilität verglichen. Die nationale Säulengrafik enthält alle Strombedarfe, Erzeugungsträger und den Außenhandel; die Mobilitätsgutschrift wird dabei nicht als zusätzliche nationale Erzeugung gezählt.

In [ ]:
# Gemeinsame GFZ-/Nutzungsverteilung bedingt auf die geometrische Klasse.
# Nutzung und GFZ werden gemeinsam gewichtet; keine gleichmäßige Aufteilung.
use_bins=joined.groupby(['Geometrie','GWR_Nutzung','GFZ'],as_index=False).zugeordnete_BGF.sum()
use_bins['s']=use_bins.zugeordnete_BGF/use_bins.groupby('Geometrie').zugeordnete_BGF.transform('sum')
np.testing.assert_allclose(use_bins.groupby('Geometrie').s.sum(),1)
selection_note='Keine vorgegebene PEQ-Auswahl. Bewertungsflächen im Dashboard auswählen; gelieferte Bestandsflächen bleiben unverändert.'
selected_area=active.loc[active.default_selected].groupby('Geometrie').BGF.sum()
allocation=use_bins.merge(geometry.reset_index(),on='Geometrie',validate='many_to_one')
allocation['D_pj']=allocation.D*allocation.s*DEFAULT['ngf_bgf']*(1-DEFAULT['saving'])
allocation['P_pj']=allocation.P*allocation.s*DEFAULT['ngf_bgf']*DEFAULT['pv_scale']
allocation['selected_BGF']=allocation.Geometrie.map(selected_area).fillna(0)*allocation.s
allocation['target_pj']=allocation.selected_BGF*(np.minimum(DEFAULT['density_f_res']/(allocation.GFZ+DEFAULT['density_dx'])-DEFAULT['density_f_ed'],DEFAULT['density_cutoff']))*3.6e-9
selected_energy=active.loc[active.default_selected].groupby('Geometrie')[['D','P']].sum()
allocation['selected_balance_pj']=(allocation.Geometrie.map(selected_energy.P).fillna(0)*DEFAULT['pv_scale']-allocation.Geometrie.map(selected_energy.D).fillna(0)*(1-DEFAULT['saving']))*DEFAULT['ngf_bgf']*allocation.s
allocation['C_pj']=allocation.target_pj-allocation.selected_balance_pj
# Ausgewählte Typen erreichen exakt den Zielsaldo; andere bleiben unverändert.
np.testing.assert_allclose((allocation.selected_balance_pj+allocation.C_pj).sum(),allocation.target_pj.sum())
np.testing.assert_allclose([allocation.D_pj.sum(),allocation.P_pj.sum()],[reference['demand'],reference['pv']],rtol=1e-12)
gfz_reference=dict(demand=allocation.D_pj.sum(),pv=allocation.P_pj.sum(),virtual=allocation.C_pj.sum(),balance=(allocation.P_pj-allocation.D_pj).sum(),corrected=(allocation.P_pj-allocation.D_pj+allocation.C_pj).sum())
np.testing.assert_allclose(gfz_reference['balance'],reference['balance'])
display(pd.DataFrame([gfz_reference],index=['GFZ-Aggregation; PJ/a bzw. PJ-Äquivalent/a']).T)
sys.path.insert(0,str(ROOT))
from styles.plotly_formatting import GFZ_USE_COLORS, COLORS
assert set(use_bins.GWR_Nutzung).issubset(GFZ_USE_COLORS), "Nutzungsfarbe fehlt"
payload.update(types=active[['Archetype','Geometrie','Huelle','Heizung','PV_Klasse','Flexibilitaet','Wetter','Masse','BGF','D','P','default_selected']].to_dict('records'),
               use_bins=use_bins[['Geometrie','GWR_Nutzung','GFZ','s']].to_dict('records'),
               use_colors=GFZ_USE_COLORS,selection_note=selection_note,gfz_reference=gfz_reference)

# BGF-gewichtete Zielsaldo-Verteilung je Geometrie (keine ungewichteten Quantile).
target_distribution=[]
for name,group in bins.groupby('Geometrie'):
    values=np.minimum(DEFAULT['density_f_res']/(group.GFZ.to_numpy()+DEFAULT['density_dx'])-DEFAULT['density_f_ed'],DEFAULT['density_cutoff'])
    weights=group.s.to_numpy(); order=np.argsort(values)
    values=values[order]; weights=weights[order]; cumulative=np.cumsum(weights)/weights.sum()
    quantiles=[float(values[min(np.searchsorted(cumulative,q),len(values)-1)]) for q in [.1,.25,.5,.75,.9]]
    target_distribution.append(dict(Geometrie=name,q10=quantiles[0],q25=quantiles[1],median=quantiles[2],q75=quantiles[3],q90=quantiles[4],mean=float(np.sum(values*weights)/weights.sum())))
display(pd.DataFrame(target_distribution).set_index('Geometrie'))
payload.update(energy_colors=COLORS,target_distribution_reference=target_distribution)

# Drei spezifische Salden je Geometrie, bezogen auf die jeweilige Gruppen-BGF.
group_balances=[]
for geo,group in active.groupby('Geometrie'):
    other=group.loc[~group.default_selected]; chosen=group.loc[group.default_selected]
    def specific_balance(part):
        area=part.BGF.sum()
        return float(((part.P*DEFAULT['pv_scale']-part.D*(1-DEFAULT['saving']))*DEFAULT['ngf_bgf']).sum()/(area*3.6e-9)) if area>0 else None
    weights=bins.loc[bins.Geometrie.eq(geo)]
    target=float((weights.s*np.minimum(DEFAULT['density_f_res']/(weights.GFZ+DEFAULT['density_dx'])-DEFAULT['density_f_ed'],DEFAULT['density_cutoff'])).sum()) if chosen.BGF.sum()>0 else None
    group_balances.append(dict(geometry=geo,non_peq_bgf_m2=float(other.BGF.sum()),peq_bgf_m2=float(chosen.BGF.sum()),non_peq=specific_balance(other),peq_typological=specific_balance(chosen),peq_target=target))
display(pd.DataFrame(group_balances).set_index('geometry'))
payload['group_balance_reference']=group_balances


## Dashboard: verändern, vergleichen, sichern

Die Startwerte entsprechen dem dokumentierten Rechenbeispiel, **keiner normativen Freigabe**.
Gebäude-PV/ÖPNV werden jeweils als Restanteile berechnet. „Übriger Verkehr“ bleibt
eigenständig. Grün/Blau kennzeichnen Erzeugung und Bedarf, Orange den Zielwert,
Violett virtuelle Kontextwerte. Alle Diagramme verwenden die aktuelle Parametrierung.

Die angezeigte zusätzliche PV-Lücke ist ein algebraischer Rest bei konstantem Bedarf,
kein Standort-, Flächen- oder Machbarkeitsnachweis. Die Dichtekurve darf verändert werden,
aber wird nicht automatisch optimiert. Die finale Publikation verwendet weiterhin nur
die bewusst übernommenen wissenschaftlichen Modellstände.


Die Kennzahlen stehen in drei Spalten: **Bedarf, Deckung und Saldo**. Gebäudebetrieb und zugeordnete Mobilität addieren sich spaltenweise zur Bottom-Up-Summe. Darunter steht der Top-Down-Vergleich mit identischer MIV-Zuordnung. Hover-Erklärungen an Reglern und Kacheln nennen Bezugsgrößen und Rechenregeln. TD bezeichnet das nationale Top-Down-Szenario.

In [ ]:
# Technische HTML-Darstellung; die wissenschaftliche Referenzrechnung steht oben.
template=(BASE/'dashboard_template.html').read_text(encoding='utf-8')
template=template.replace('__DENSITY_SCRIPT__',(BASE/'dashboard_density.js').read_text(encoding='utf-8'))
encoded=json.dumps(payload,ensure_ascii=False,allow_nan=False).replace('<',r'\u003c')
display(HTML(template.replace('__PAYLOAD__',encoded)))

(BASE/'outputs/dashboard_data.json').write_text(encoded,encoding='utf-8')
